# PLR MLP (딥러닝) — 시드 7

- 모델: PLR 주기 임베딩 + MLP (512-256)
- 검증: 5-fold 교차검증 (시드 7), Macro F1
- 참고: 시드 3개 평균은 0.5271 (42: 0.5300 / 7: 0.5418 / 123: 0.5094)

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

if os.path.basename(os.getcwd()) == "model":
    os.chdir("../학생배포")
print("현재 실행 위치:", os.getcwd())

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)  # 결과 재현용

현재 실행 위치: c:\Users\apxkz\OneDrive\Desktop\final_project\학생배포


## 1. 데이터 불러오기 · 전처리

In [2]:
X = pd.read_csv("data/주조공정_입력.csv", header=1)
X.columns = X.columns.str.strip()                       # 컬럼명 공백 제거
split = pd.read_csv("분할표.csv", encoding="utf-8-sig")
label = pd.read_csv("정답_train.csv", encoding="utf-8-sig")

train = (X.merge(split, on="shot_id").query("split == 'train'")
          .merge(label, on="shot_id").sort_values("shot_id").reset_index(drop=True))

ID_COLS = ["shot_id", "id", "Shot"]                       # 식별열: 피처 금지
CONST_COLS = [c for c in X.columns if X[c].nunique() <= 1]  # 값이 하나뿐인 열 8개
FEATS = [c for c in X.columns if c not in ID_COLS + CONST_COLS]

X_train = train[FEATS]
y_train = train["label"]
print("학습 데이터:", X_train.shape, "| 피처", len(FEATS), "개")

학습 데이터: (5274, 21) | 피처 21 개


## 2. 모델 정의

In [3]:
class Periodic(nn.Module):
    """PLR 임베딩: 변수마다 sin·cos 16쌍 → 변수별 Linear(32→16) + ReLU"""
    def __init__(self, n_feats, k=16, d=16, sigma=0.3):
        super().__init__()
        self.c = nn.Parameter(torch.randn(n_feats, k) * sigma)        # 주파수 (학습됨)
        self.W = nn.Parameter(torch.randn(n_feats, 2 * k, d) / np.sqrt(2 * k))
        self.b = nn.Parameter(torch.zeros(n_feats, d))

    def forward(self, x):
        z = 2 * np.pi * x[:, :, None] * self.c[None]
        z = torch.cat([torch.sin(z), torch.cos(z)], dim=-1)
        return torch.relu(torch.einsum("bfk,fkd->bfd", z, self.W) + self.b).flatten(1)


class PLRMLP:
    """결측 중앙값 → StandardScaler → PLR → 512 → 256 → 7"""
    def __init__(self, seed=SEED, epochs=300, batch=128, lr=2e-3):
        self.seed, self.epochs, self.batch, self.lr = seed, epochs, batch, lr

    def _to_tensor(self, X):
        return torch.tensor(self.scaler.transform(self.imputer.transform(X)), dtype=torch.float32)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)
        self.classes_ = np.array(sorted(set(y)))
        y_t = torch.tensor(np.searchsorted(self.classes_, np.asarray(y)))

        self.imputer = SimpleImputer(strategy="median").fit(X)
        self.scaler = StandardScaler().fit(self.imputer.transform(X))
        X_t = self._to_tensor(X)
        n_feats = X_t.shape[1]

        self.net = nn.Sequential(
            Periodic(n_feats),
            nn.Linear(n_feats * 16, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, len(self.classes_)),
        )
        loss_fn = nn.CrossEntropyLoss()
        opt = torch.optim.AdamW(self.net.parameters(), lr=self.lr, weight_decay=0.0)
        n_batches = (len(X_t) + self.batch - 1) // self.batch
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=self.lr, total_steps=self.epochs * n_batches)

        # 조기 종료 없이 끝까지 학습
        for _ in range(self.epochs):
            self.net.train()
            perm = torch.randperm(len(X_t))
            for i in range(0, len(X_t), self.batch):
                b = perm[i:i + self.batch]
                opt.zero_grad()
                loss_fn(self.net(X_t[b]), y_t[b]).backward()
                opt.step()
                sched.step()
        return self

    def predict(self, X):
        self.net.eval()
        with torch.no_grad():
            return self.classes_[self.net(self._to_tensor(X)).argmax(1).numpy()]

## 3. 5-fold 교차검증 (시드 7)

fold마다 1~2분 정도 걸려요.

In [4]:
oof = np.empty(len(y_train), dtype=object)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), 1):
    model = PLRMLP(seed=SEED).fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    oof[va_idx] = model.predict(X_train.iloc[va_idx])
    score = f1_score(y_train.iloc[va_idx], oof[va_idx], average="macro")
    print(f"fold {fold}: Macro F1 {score:.4f}")

print(f"\n전체 Macro F1 (시드 {SEED}): {f1_score(y_train, oof, average='macro'):.4f}")

fold 1: Macro F1 0.5357
fold 2: Macro F1 0.5094
fold 3: Macro F1 0.5160
fold 4: Macro F1 0.5758
fold 5: Macro F1 0.5401

전체 Macro F1 (시드 7): 0.5373


## 4. 결과

In [5]:
print(classification_report(y_train, oof, digits=3, zero_division=0))

classes = sorted(y_train.unique())
cm = pd.DataFrame(confusion_matrix(y_train, oof, labels=classes), index=classes, columns=classes)
cm.index.name = "실제 \\ 예측"
cm

                precision    recall  f1-score   support

          기타불량      0.250     0.132     0.172        76
  불량_Blow_Hole      0.693     0.639     0.665       233
불량_Deformation      0.333     0.255     0.289       106
불량_Exfoliation      0.520     0.389     0.445       198
 불량_Short_Shot      0.572     0.548     0.560       469
      불량_Stain      0.833     0.650     0.730       100
            정상      0.881     0.918     0.899      4092

      accuracy                          0.823      5274
     macro avg      0.583     0.504     0.537      5274
  weighted avg      0.811     0.823     0.816      5274



,기타불량,불량_Blow_Hole,불량_Deformation,불량_Exfoliation,불량_Short_Shot,불량_Stain,정상
실제 \ 예측,,,,,,,
기타불량,10,0,2,2,5,0,57
불량_Blow_Hole,0,149,0,0,7,1,76
불량_Deformation,1,0,27,9,4,0,65
불량_Exfoliation,2,0,3,77,11,0,105
불량_Short_Shot,2,5,6,12,257,5,182
불량_Stain,1,3,0,0,9,65,22
정상,24,58,43,48,156,7,3756
